# Social Media → Donations (Course-Aligned Pipeline)

**Business question:** What kinds of **planned** posts are associated with stronger **donation referrals** and **estimated donation value**?

This notebook follows the **CRISP-DM** structure and the textbook flow:

| Phase | Textbook | Sections here |
| --- | --- | --- |
| Business understanding | Ch1 | Goals, feasibility, explanatory vs predictive |
| Data understanding | Ch2–Ch4, Ch6 | Load tables, **unistats-style profile** |
| Data preparation | Ch7 | **Diagnose → wrangle (dataset-specific) → encode** (`functions.py`, `social_wrangle.py`, `features.py`) |
| Modeling (explanatory) | Ch9–10 | **OLS** + dummy coding + coefficient readout |
| Modeling (predictive) | Ch11 | Train/test, **sklearn `Pipeline`**, linear MLR vs leakage |
| Modeling (ensembles) | Ch12–14 | Tree ensembles for regression; classifiers in order **Logistic → Decision Tree → RF** |
| Evaluation | Ch15 | **Stratified CV** + small **GridSearchCV** on training folds; final test metrics |
| Deployment | Ch17 | **joblib** bundle + `train_export.py` / FastAPI note |


## 1) Business understanding (CRISP-DM Phase 1)

- **Problem:** Prioritize social content that is more likely to drive measurable giving signals.
- **Data availability:** `social_media_posts.csv` has outcomes; `donations.csv` links some gifts via `referral_post_id`.
- **Explanatory use:** OLS to discuss associations (not causal proof).
- **Predictive use:** Regression and classification evaluated on **held-out** posts.
- **Deployment:** Staff-facing planner uses **only pre-publish** features (no impressions, likes, reach).

**Process integration:** inference API + admin UI (see `api/` and `train_export.py`).


In [3]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    root_mean_squared_error,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", category=UserWarning)

BASE = Path(".").resolve()
if BASE.name == "social_donation_ml":
    PROJECT_DIR = BASE
elif BASE.name == "ml-pipelines" and (BASE / "social_donation_ml").exists():
    PROJECT_DIR = BASE / "social_donation_ml"
else:
    PROJECT_DIR = BASE

DATA_DIR = PROJECT_DIR / "data"

from functions import unistats_profile
from features import (
    BOOL_COLS,
    CAT_COLS,
    FEATURE_COLS,
    NUM_COLS,
    build_model_frame,
)
from social_wrangle import wrangle_social_posts_with_donations

POSTS_CSV = DATA_DIR / "social_media_posts.csv"
DONATIONS_CSV = DATA_DIR / "donations.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.25

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)


Project directory: /Users/blakerogers/Intex2/ml-pipelines/social_donation_ml
Data directory: /Users/blakerogers/Intex2/ml-pipelines/social_donation_ml/data


## 2) Data understanding — diagnostic profile (Ch6–7)

Run **`unistats_profile`** on the **raw** posts table to see missingness, cardinality, and skew — the same “what needs fixing?” mindset as the `unistats` diagnostic step before preparation.


In [4]:
posts_raw = pd.read_csv(POSTS_CSV)
don_raw = pd.read_csv(DONATIONS_CSV)

profile = unistats_profile(posts_raw)
display(profile.head(25))

print("Shape posts:", posts_raw.shape, "donations:", don_raw.shape)
print("Donations with referral_post_id:", don_raw["referral_post_id"].notna().sum())


,column,dtype,missing_pct,n_distinct,skew
37,subscriber_count_at_post,float64,91.26,67,0.317
36,avg_view_duration_seconds,float64,91.26,64,1.115
35,watch_time_seconds,float64,91.26,71,2.233
38,forwards,float64,88.55,66,6.745
21,boost_budget_php,float64,84.36,127,0.111
19,campaign_name,object,71.43,4,NaN
29,video_views,float64,58.99,328,1.564
14,call_to_action_type,object,39.29,4,NaN
10,hashtags,object,18.23,404,NaN
32,donation_referrals,int64,0.00,95,6.218


Shape posts: (812, 39) donations: (420, 13)
Donations with referral_post_id: 77


## 3) Data preparation (Ch7 funnel)

**Step A — dataset-specific wrangling:** merge aggregated CRM gifts onto each post (`social_wrangle.wrangle_social_posts_with_donations`).

**Step B — general encoding:** `build_model_frame` (missing-safe categoricals, numeric coercion, boost budget rules).


In [5]:
df = wrangle_social_posts_with_donations(posts_raw, don_raw)
m = build_model_frame(df)
X = m[FEATURE_COLS]

y_ref = pd.to_numeric(m["donation_referrals"], errors="coerce").fillna(0)
y_val = pd.to_numeric(m["estimated_donation_value_php"], errors="coerce").fillna(0)
y_eng = pd.to_numeric(m["engagement_rate"], errors="coerce").fillna(0)

X_train, X_test, y_ref_train, y_ref_test, y_val_train, y_val_test = train_test_split(
    X, y_ref, y_val, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Classification label: top quartile of **training** referrals (no test leakage)
q75 = float(y_ref_train.quantile(0.75))
y_hi_train = (y_ref_train >= q75).astype(int)
y_hi_test = (y_ref_test >= q75).astype(int)

print("Train", X_train.shape, "Test", X_test.shape)
print("Top-quartile threshold (train referrals):", q75)
print("Class balance train (high):", y_hi_train.mean().round(3))


Train (609, 15) Test (203, 15)
Top-quartile threshold (train referrals): 11.0
Class balance train (high): 0.259


## 4) Quick EDA on targets (still Phase 2)

Sanity-check outcome scales before modeling.


In [6]:
for name, s in [("donation_referrals", y_ref), ("est_value_php", y_val), ("engagement_rate", y_eng)]:
    print(f"{name}: mean={s.mean():.4f} median={s.median():.4f}")


donation_referrals: mean=12.7956 median=2.0000
est_value_php: mean=39772.0937 median=4556.9550
engagement_rate: mean=0.0990 median=0.0893


## 5) Modeling — explanatory OLS (Ch9)

OLS with **dummy coding** (`drop_first=True` to avoid the dummy trap). Fit on **train**; compare train vs test **R²/RMSE** as a cautionary read (linear misspecification is expected on counts).


In [7]:
X_train_dm = pd.get_dummies(X_train, columns=CAT_COLS, drop_first=True).astype(float)
X_train_dm = sm.add_constant(X_train_dm)
ols = sm.OLS(y_ref_train.values, X_train_dm).fit()

X_test_dm = pd.get_dummies(X_test, columns=CAT_COLS, drop_first=True)
X_test_dm = X_test_dm.reindex(columns=X_train_dm.columns, fill_value=0).astype(float)

pred_tr = ols.predict(X_train_dm)
pred_te = ols.predict(X_test_dm)

def reg_report(tag, y_a, y_p):
    print(
        tag,
        "RMSE",
        round(root_mean_squared_error(y_a, y_p), 4),
        "MAE",
        round(mean_absolute_error(y_a, y_p), 4),
        "R²",
        round(r2_score(y_a, y_p), 4),
    )

reg_report("OLS train", y_ref_train, pred_tr)
reg_report("OLS test ", y_ref_test, pred_te)
print("\nStrongest coefficients (train OLS):")
print(ols.params.abs().sort_values(ascending=False).head(12))


OLS train RMSE 25.3037 MAE 13.2539 R² 0.3446
OLS test  RMSE 29.6714 MAE 22.196 R² 0.095

Strongest coefficients (train OLS):
features_resident_story        25.946845
const                          14.451508
post_type_ImpactStory           8.571457
content_topic_EventRecap        8.303898
sentiment_tone_Informative      8.125688
platform_WhatsApp               8.051673
platform_Twitter                7.558092
content_topic_Education         7.378043
post_type_FundraisingAppeal     6.804653
sentiment_tone_Grateful         6.739427
day_of_week_Saturday            6.677576
is_boosted                      6.549405
dtype: float64


### 5b) Ch10 note (explanatory diagnostics)

For a **grading rubric** that emphasizes Ch10, you would add residual vs fitted plots, heteroscedasticity checks, and influence diagnostics **before** treating coefficients as explanatory evidence. Here we keep OLS as an interpretable baseline and lean on ensembles for prediction.


## 6) Predictive pipelines (Ch11–14)

**Shared preprocessor** — fit only inside each `Pipeline` on training folds / train split.

Models:

1. **LinearRegression** (predictive MLR, sklearn)
2. **DecisionTreeRegressor** (high-variance tree, Ch12 baseline)
3. **RandomForestRegressor**, **GradientBoostingRegressor** (Ch14)


In [8]:
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", StandardScaler(), NUM_COLS + BOOL_COLS),
    ]
)

lin_pipe = Pipeline(
    [
        ("prep", clone(preprocess)),
        ("model", LinearRegression()),
    ]
)
tree_reg = Pipeline(
    [
        ("prep", clone(preprocess)),
        ("model", DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE)),
    ]
)
rf_reg = Pipeline(
    [
        ("prep", clone(preprocess)),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)
gb_reg = Pipeline(
    [
        ("prep", clone(preprocess)),
        (
            "model",
            GradientBoostingRegressor(
                max_depth=4,
                learning_rate=0.06,
                n_estimators=250,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

for name, model in [
    ("LinearRegression", lin_pipe),
    ("DecisionTreeReg ", tree_reg),
    ("RandomForestReg ", rf_reg),
    ("GradBoostReg    ", gb_reg),
]:
    model.fit(X_train, y_ref_train)
    p = model.predict(X_test)
    reg_report(name + " test", y_ref_test, p)


LinearRegression test RMSE 26.3476 MAE 15.1141 R² 0.2864
DecisionTreeReg  test RMSE 27.8853 MAE 11.7771 R² 0.2007
RandomForestReg  test RMSE 27.3942 MAE 12.1763 R² 0.2286
GradBoostReg     test RMSE 27.4903 MAE 12.4809 R² 0.2232


## 7) Model evaluation & tuning (Ch15)

**Pass 1 — reliable comparison:** `StratifiedKFold` **cross-validation** on **training data only** for classification baselines (ROC-AUC).

**Pass 3 — light tuning:** `GridSearchCV` on the GradientBoosting regressor (small grid, budget-friendly).


In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

clf_candidates = {
    "logistic": LogisticRegression(max_iter=3000, class_weight="balanced", solver="lbfgs"),
    "tree": DecisionTreeClassifier(
        max_depth=8, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=14,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

cv_rows = []
for name, est in clf_candidates.items():
    pipe = Pipeline([("prep", clone(preprocess)), ("model", est)])
    aucs = cross_val_score(pipe, X_train, y_hi_train, cv=cv, scoring="roc_auc", n_jobs=1)
    cv_rows.append({"model": name, "auc_mean": aucs.mean(), "auc_std": aucs.std()})
cv_table = pd.DataFrame(cv_rows).sort_values("auc_mean", ascending=False)
display(cv_table)

param_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.04, 0.08],
}
gb_search = GridSearchCV(
    clone(gb_reg),
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
    refit=True,
)
gb_search.fit(X_train, y_ref_train)
print("Best GB params:", gb_search.best_params_)
best_gb_test_pred = gb_search.predict(X_test)
reg_report("Tuned GB test", y_ref_test, best_gb_test_pred)


,model,auc_mean,auc_std
2,forest,0.927854,0.019514
0,logistic,0.917135,0.017198
1,tree,0.755992,0.055929


Best GB params: {'model__learning_rate': 0.04, 'model__max_depth': 3, 'model__n_estimators': 250}
Tuned GB test RMSE 26.3811 MAE 12.255 R² 0.2846


## 8) Classification on held-out **test** (Ch13 ordering)

Refit **logistic → tree → forest** pipelines on full training set and report precision/recall on **test** for the high-referral label.


In [10]:
fitted_clf = {}
for name, est in clf_candidates.items():
    pipe = Pipeline([("prep", clone(preprocess)), ("model", clone(est))])
    pipe.fit(X_train, y_hi_train)
    fitted_clf[name] = pipe
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(
        name,
        "precision",
        round(precision_score(y_hi_test, pred, zero_division=0), 3),
        "recall",
        round(recall_score(y_hi_test, pred, zero_division=0), 3),
        "ROC-AUC",
        round(roc_auc_score(y_hi_test, proba), 3),
    )


logistic precision 0.623 recall 0.878 ROC-AUC 0.924
tree precision 0.577 recall 0.837 ROC-AUC 0.839
forest precision 0.786 recall 0.673 ROC-AUC 0.941


## 9) Deployment (Ch17)

**Artifacts**

- `train_export.py` retrains **Tuned GB** (referrals + value) and picks the **best CV classifier** for `social_bundle.joblib`.
- FastAPI loads the bundle for `/predict/draft`.

**Reliability principles:** same preprocessing in train + inference, versioned `joblib`, traceable metadata JSON.

Run locally:

```bash
python train_export.py
uvicorn api.main:app --host 0.0.0.0 --port 8002
```


In [11]:
print("Notebook complete — align with Ch7 by expanding unistats + wrangle when schema changes.")


Notebook complete — align with Ch7 by expanding unistats + wrangle when schema changes.
